# 02 — Training (D3–D6)

**YOLOv8s / YOLO11s / YOLO26s under identical settings.**

The paper's entire claim is *identical conditions*. Everything comes from `src/config.py`. **Do not tune per model** — if you pass a different lr for one model, the controlled comparison is dead and the contribution goes with it.

**Settings:** accelerator **GPU P100** (see §0), internet ON.

**Gate D4:** if YOLOv8s mAP@0.4 < 0.20 on positive-only CV, labels or fusion are broken — go back to notebook 01. Do not launch two more models on bad data.

## 0. Why P100, not T4×2

Kaggle bills GPU sessions by **wall-clock, not per-GPU** — selecting `T4 x2` and using `device=0` burns identical quota for strictly less card. So the real choice is single P100 vs single T4.

| | P100 | T4 |
|---|---|---|
| Bandwidth | **732 GB/s** | 320 GB/s |
| FP32 | 9.5 TFLOPS | 8.1 TFLOPS |
| FP16 | 19 TFLOPS (no tensor cores) | 65 TFLOPS (tensor cores) |
| Power | 250 W | **70 W** (throttles) |

T4 wins on FP16 paper specs; reality is closer because T4 is bandwidth-starved and power-capped. Expect **P100 ≈ 1.0–1.3× a single T4** here.

**Don't spend deadline time on 2×T4 DDP** — notebook spawn failures are a known sink. Best case saves ~2 h; worst case eats a day.

In [ ]:
!pip install -q ultralytics wandb

import sys
REPO = "/kaggle/input/vindr-iccit-repo"
sys.path.insert(0, REPO)

from pathlib import Path
from src import config as C
from src.training import train as T
from src.utils.run_logger import RunLogger, benchmark_inference, capture_gpu

# Confirm the card BEFORE anything else -- this value goes in the paper.
capture_gpu()

In [ ]:
# Point at the dataset built by notebook 01 (attached as an input dataset).
DATA_YAML = Path("/kaggle/input/vindr-yolo-prepared/data.yaml")
assert DATA_YAML.exists(), "attach the notebook-01 output as an input dataset"
print(DATA_YAML.read_text())

## 1. W&B overhead measurement (D3, ~10 min)

Run 5 epochs with and without W&B, compare `sec_per_epoch` from `RunLogger`.

General figure is 1–3% (async background process; **image logging is the expensive part, not scalars**). Measure it on *your* hardware and network rather than trusting that. If it comes back >5%, disable image logging and keep the scalars.

In [ ]:
# Smoke test WITHOUT wandb. Also validates the whole pipeline end to end.
_ = T.train_one("yolov8s", DATA_YAML, epochs=5, use_wandb=False,
                runs_dir=C.RUNS_DIR / "smoke_nowandb")

In [ ]:
# Same, WITH wandb. Needs WANDB_API_KEY in Kaggle Secrets.
from kaggle_secrets import UserSecretsClient
import os
os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")

_ = T.train_one("yolov8s", DATA_YAML, epochs=5, use_wandb=True,
                runs_dir=C.RUNS_DIR / "smoke_wandb")

In [ ]:
a = RunLogger.master_table(C.RUNS_DIR / "smoke_nowandb").sec_per_epoch.iloc[-1]
b = RunLogger.master_table(C.RUNS_DIR / "smoke_wandb").sec_per_epoch.iloc[-1]
print(f"no-wandb {a:.1f}s/epoch | wandb {b:.1f}s/epoch | overhead {(b/a-1)*100:.1f}%")

## 2. Full training runs (D3–D5)

Sequential, one GPU. ~1.5–2 h per model on positive-only at 512px.

**Checkpoints save every epoch.** The 12h session cap will interrupt you at some point — re-run the cell with `resume=True` on the affected model.

In [ ]:
results = T.train_all(DATA_YAML, use_wandb=True)
{k: v["status"] for k, v in results.items()}

In [ ]:
# If a session was killed mid-run, resume just that model:
# _ = T.train_one("yolo26s", DATA_YAML, resume=True)

## 3. mAP@0.4 — the competition metric

> ⚠️ **Ultralytics cannot give you mAP@0.4.** It reports @0.5 and @0.5:0.95 only. The VinBigData competition used IoU > 0.4.
>
> If @0.5 silently lands in a column labelled @0.4, your entire related-work comparison misaligns — and that error survives to camera-ready.

**Report both:** @0.4 for the competition lineage (0.253 / 0.314), @0.5 for the modern literature (0.338 / 0.415 / 0.453).

In [ ]:
from ultralytics import YOLO
from src.evaluation import metrics as M

DATA_ROOT = DATA_YAML.parent
weights = T.all_best_weights()

evals = {}
for key, w in weights.items():
    print(f"\n=== {key} ===")
    evals[key] = M.evaluate_full(
        YOLO(w),
        DATA_ROOT / "images" / "test",
        DATA_ROOT / "labels" / "test",
        imgsz=C.IMGSZ,
    )

In [ ]:
import pandas as pd

summary = pd.DataFrame({
    k: {"mAP@0.4": v["map40"], "mAP@0.5": v["map50"]} for k, v in evals.items()
}).T.round(4)

print(summary)
print("\nPublished bar (do NOT frame as beating these):")
for name, vals in C.PUBLISHED_BASELINES.items():
    print(f"  {name}: {vals}")

### ⛔ Gate D4

YOLOv8s mAP@0.4 **< 0.20** → labels or fusion are broken, not the model. The reference hit ~0.45 CV positive-only with EfficientDet. Go back to notebook 01.

In [ ]:
m = evals["yolov8s"]["map40"]
if m < 0.20:
    print(f"⛔ GATE FAILED: mAP@0.4={m:.4f} < 0.20. Debug data pipeline.")
else:
    print(f"✓ gate passed: mAP@0.4={m:.4f}")

## 4. Inference benchmark (D10) — same card, one session

> ⚠️ **mAP is hardware-independent. Latency is not.**
>
> YOLO26 is marketed as edge-first (claimed 43% CPU speed gain over YOLO11), so a latency table is expected. It is only valid if produced **here** — all three checkpoints back to back, one session, one card — not stitched from training runs that may have landed on different GPUs.

In [ ]:
RunLogger.check_hardware_consistency(C.RUNS_DIR)
bench = benchmark_inference(weights, imgsz=C.IMGSZ, out_dir=str(C.RUNS_DIR))
pd.DataFrame(bench).T[["ms_per_image", "fps", "n_params_M", "gpu_family"]]

In [ ]:
# Persist everything -- /kaggle/working dies with the session.
import json
(C.RUNS_DIR / "eval_summary.json").write_text(json.dumps(
    {k: {"map40": v["map40"], "map50": v["map50"],
         "per_class_40": v["detail_40"]["per_class"]} for k, v in evals.items()},
    indent=2, default=str))
print("Save Version → Save Output, then attach to notebook 03")